# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score, 
    mean_squared_error, 
    mean_absolute_error, 
    mean_absolute_percentage_error
)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(12345)

# 2. Read Data

In [ ]:
data = pd.read_excel('rice_dataset.xlsx')

data

# 3. Clean Data

## 3.1. Handle Missing Values

### 3.1.1. Check Missing Values

In [ ]:
data.info()

### 3.1.2. Remove Missing Values (Listwise Deletion)

In [ ]:
# no NaN to remove
# data.dropna(axis=0, inplace=True)

## 3.2. Handle Outliers

### 3.2.1. Check Outliers

In [ ]:
data.describe()

### 3.2.2. Remove Outliers

In [ ]:
# no outlier to remove

# 4. Split Dataset into Training & Test Sets

In [ ]:
target_name = 'WATER REQUIREMENT'
feature_name = list(data.columns.drop(target_name))

In [ ]:
X = data[feature_name]
y = data[target_name]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, shuffle=True)

# 5. Data Preparation

In [ ]:
numerical_feature = []
categorical_feature = ['SOIL TYPE', 'REGION', 'TEMPERATURE', 'WEATHER CONDITION']

In [ ]:
for feature in categorical_feature:
    print(feature, ':', np.unique(X_train[feature]))

In [ ]:
ordinal_feature = ['SOIL TYPE', 'REGION', 'TEMPERATURE']
nominal_feature = ['WEATHER CONDITION']

## 5.1. Ordinal Encoding

### 5.1.1. Training Set

In [ ]:
categories = [
    np.array(['DRY', 'HUMID', 'WET'], dtype=object),
    np.array(['DESERT', 'SEMI ARID', 'SEMI HUMID', 'HUMID'], dtype=object),
    np.array(['10-20', '20-30', '30-40', '40-50'], dtype=object)
]

In [ ]:
ordinal_encoder = OrdinalEncoder(categories=categories)
X_train[ordinal_feature] = ordinal_encoder.fit_transform(X_train[ordinal_feature])

### 5.1.2. Test Set

In [ ]:
X_test[ordinal_feature] = ordinal_encoder.transform(X_test[ordinal_feature])

## 5.2. One Hot Encoding

### 5.2.1. Training Set

In [ ]:
one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
one_hot_encoder.fit(X_train[nominal_feature])

In [ ]:
one_hot_feature = []
for i, feature in enumerate(nominal_feature):
    for cate in one_hot_encoder.categories_[i]:
        one_hot_feature_name = str(feature) + '_' + str(cate)
        one_hot_feature.append(one_hot_feature_name)

In [ ]:
X_train[one_hot_feature] = one_hot_encoder.transform(X_train[nominal_feature])
X_train.drop(nominal_feature, axis=1, inplace=True)

### 5.2.2. Test Set

In [ ]:
X_test[one_hot_feature] = one_hot_encoder.transform(X_test[nominal_feature])
X_test.drop(nominal_feature, axis=1, inplace=True)

# 6. Model Creation

## 6.1. Setting Parameters

In [ ]:
reg = LinearRegression()

## 6.2. Train Model

In [ ]:
reg.fit(X_train, y_train)

## 6.3. Model's Weight & Bias

In [ ]:
reg.coef_

In [ ]:
reg.intercept_

# 7. Prediction

## 7.1. Training Set

In [ ]:
y_pred_train = reg.predict(X_train)

## 7.2. Test Set

In [ ]:
y_pred_test = reg.predict(X_test)

# 8. Model Evaluation

## 8.1. Training Set

### 8.1.1. Scoring

In [ ]:
print('r2_score =\t\t\t', r2_score(y_train, y_pred_train))
print('mean_squared_error =\t\t', mean_squared_error(y_train, y_pred_train))
print('mean_absolute_error =\t\t', mean_absolute_error(y_train, y_pred_train))
print('mean_absolute_percentage_error =', mean_absolute_percentage_error(y_train, y_pred_train))

### 8.1.2. Scatter Plot between Predicted & Actual Values

In [ ]:
plt.scatter(y_pred_train, y_train)

plt.plot(y_pred_train, y_pred_train, color='red')

plt.title('Scatter Plot between Predicted & Actual Values')
plt.xlabel('Predicted')
plt.ylabel('Actual')

## 8.2. Test Set

### 8.2.1. Scoring

In [ ]:
print('r2_score =\t\t\t', r2_score(y_test, y_pred_test))
print('mean_squared_error =\t\t', mean_squared_error(y_test, y_pred_test))
print('mean_absolute_error =\t\t', mean_absolute_error(y_test, y_pred_test))
print('mean_absolute_percentage_error =', mean_absolute_percentage_error(y_test, y_pred_test))

### 8.2.2. Scatter Plot between Predicted & Actual Values

In [ ]:
plt.scatter(y_pred_test, y_test)

plt.plot(y_pred_test, y_pred_test, color='red')

plt.title('Scatter Plot between Predicted & Actual Values')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# 9. Save Model

In [ ]:
import pickle

In [ ]:
pickle.dump((reg, 
             ordinal_encoder, 
             one_hot_encoder, 
             feature_name, 
             numerical_feature, 
             ordinal_feature, 
             nominal_feature), 
            open('rice_model.pickle', 'wb'))